In [17]:
import subprocess, time, requests

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

sh("kubectl get nodes -o wide");

NAME           STATUS   ROLES           AGE    VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION             CONTAINER-RUNTIME
minikube       Ready    control-plane   165m   v1.37.0   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   7.0.0-30-generic (arm64)   containerd://2.3.4
minikube-m02   Ready    <none>          165m   v1.37.0   192.168.49.3   <none>        Debian GNU/Linux 12 (bookworm)   7.0.0-30-generic (arm64)   containerd://2.3.4



In [18]:
!docker tag spam-api:multistage spam-api:v1
!minikube image load spam-api:v1

In [19]:
sh("minikube ssh --node minikube \"sudo crictl images | grep spam-api\"")
sh("minikube ssh --node minikube-m02 \"sudo crictl images | grep spam-api\"");

docker.io/library/spam-api                v1                   4cb3b7b71c775       148MB
docker.io/library/spam-api                v2                   33c4bd87c6a65       145MB

docker.io/library/spam-api                v1                   4cb3b7b71c775       148MB
docker.io/library/spam-api                v2                   33c4bd87c6a65       145MB



In [20]:
%%writefile deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: spam-api
  labels:
    app: spam-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: spam-api
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxUnavailable: 0
      maxSurge: 1
  template:
    metadata:
      labels:
        app: spam-api
    spec:
      containers:
      - name: api
        image: spam-api:v1
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 8080
        resources:
          requests:
            cpu: "250m"
            memory: "256Mi"
          limits:
            cpu: "500m"
            memory: "512Mi"
        readinessProbe:
          httpGet:
            path: /healthz
            port: 8080
          initialDelaySeconds: 3
          periodSeconds: 5
        lifecycle:
          preStop:
            exec:
              command: ["sleep", "5"]

Overwriting deployment.yaml


In [21]:
%%writefile service.yaml
apiVersion: v1
kind: Service
metadata:
  name: spam-api-svc
spec:
  type: NodePort
  selector:
    app: spam-api
  ports:
  - port: 80
    targetPort: 8080
    nodePort: 30080

Overwriting service.yaml


In [23]:
sh("kubectl delete deployment spam-api --ignore-not-found --wait=true")
sh("kubectl apply -f deployment.yaml")
sh("kubectl apply -f service.yaml")
sh("kubectl rollout status deployment/spam-api --timeout=120s")
sh('kubectl annotate deployment spam-api kubernetes.io/change-cause="v1: Q1 multi-stage image" --overwrite')
sh("kubectl get deployment spam-api")
sh("kubectl get pods -l app=spam-api -o wide")
sh("kubectl get svc spam-api-svc");

deployment.apps "spam-api" deleted from default namespace

deployment.apps/spam-api created

service/spam-api-svc unchanged

Waiting for deployment "spam-api" rollout to finish: 0 of 2 updated replicas are available...
Waiting for deployment "spam-api" rollout to finish: 1 of 2 updated replicas are available...
deployment "spam-api" successfully rolled out

deployment.apps/spam-api annotated

NAME       READY   UP-TO-DATE   AVAILABLE   AGE
spam-api   2/2     2            2           7s

NAME                        READY   STATUS    RESTARTS   AGE   IP            NODE           NOMINATED NODE   READINESS GATES
spam-api-79965dd7d4-5g2d4   1/1     Running   0          7s    10.244.0.12   minikube       <none>           <none>
spam-api-79965dd7d4-99jvc   1/1     Running   0          7s    10.244.1.10   minikube-m02   <none>           <none>

NAME           TYPE       CLUSTER-IP       EXTERNAL-IP   PORT(S)        AGE
spam-api-svc   NodePort   10.103.123.116   <none>        80:30080/TCP   87

In [24]:
BASE = f"http://{sh('minikube ip').stdout.strip()}:30080"

health = requests.get(f"{BASE}/healthz")
print("Health check:", health.status_code, health.json())

for text in ["WIN a FREE iPhone now! Click here: bit.ly/xyz123",
             "Hey, are we still meeting for lunch on Friday?"]:
    pred = requests.post(f"{BASE}/predict", json={"text": text}).json()
    print(text, "->", pred)

192.168.49.2

Health check: 200 {'status': 'ok'}
WIN a FREE iPhone now! Click here: bit.ly/xyz123 -> {'label': 'spam'}
Hey, are we still meeting for lunch on Friday? -> {'label': 'ham'}


In [27]:
print("before:")
sh("kubectl get pods -l app=spam-api -o wide")

pod_to_delete = sh("kubectl get pods -l app=spam-api -o jsonpath='{.items[0].metadata.name}'").stdout.strip()
print("deleting pod:", pod_to_delete, "\n")
sh(f"kubectl delete pod {pod_to_delete} --wait=false")

time.sleep(1)
print("1 sec later:")
sh("kubectl get pods -l app=spam-api -o wide")

print("service still answering:", requests.get(f"{BASE}/healthz").status_code, "\n")

time.sleep(15)
print("15 secs later:")
sh("kubectl get pods -l app=spam-api -o wide");

before:
NAME                        READY   STATUS    RESTARTS   AGE   IP            NODE           NOMINATED NODE   READINESS GATES
spam-api-79965dd7d4-d95v4   1/1     Running   0          17s   10.244.1.11   minikube-m02   <none>           <none>
spam-api-79965dd7d4-l5s8x   1/1     Running   0          48s   10.244.0.13   minikube       <none>           <none>

spam-api-79965dd7d4-d95v4
deleting pod: spam-api-79965dd7d4-d95v4 

pod "spam-api-79965dd7d4-d95v4" deleted from default namespace

1 sec later:
NAME                        READY   STATUS        RESTARTS   AGE   IP            NODE           NOMINATED NODE   READINESS GATES
spam-api-79965dd7d4-d95v4   1/1     Terminating   0          18s   10.244.1.11   minikube-m02   <none>           <none>
spam-api-79965dd7d4-l5s8x   1/1     Running       0          49s   10.244.0.13   minikube       <none>           <none>
spam-api-79965dd7d4-sqc4r   0/1     Running       0          1s    10.244.1.12   minikube-m02   <none>           <none>


In [28]:
sh("kubectl get deployment,replicaset -l app=spam-api")
sh("kubectl describe replicaset -l app=spam-api | grep -A 10 Events");

NAME                       READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/spam-api   2/2     2            2           87s

NAME                                  DESIRED   CURRENT   READY   AGE
replicaset.apps/spam-api-79965dd7d4   2         2         2       87s

Events:
  Type    Reason            Age   From                   Message
  ----    ------            ----  ----                   -------
  Normal  SuccessfulCreate  87s   replicaset-controller  Created pod: spam-api-79965dd7d4-99jvc
  Normal  SuccessfulCreate  87s   replicaset-controller  Created pod: spam-api-79965dd7d4-5g2d4
  Normal  SuccessfulCreate  72s   replicaset-controller  Created pod: spam-api-79965dd7d4-l5s8x
  Normal  SuccessfulCreate  41s   replicaset-controller  Created pod: spam-api-79965dd7d4-d95v4
  Normal  SuccessfulCreate  24s   replicaset-controller  Created pod: spam-api-79965dd7d4-sqc4r



In [29]:
%%writefile spam_app.py
# Spam detection API v2, /healthz now reports the version
import os
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

APP_VERSION = "v2"
MODEL_PATH = os.environ.get("MODEL_PATH", "data/model.joblib")

app = FastAPI(title="Spam Detection API")
_model = None


@app.on_event("startup")
def load_model():
    global _model
    _model = joblib.load(MODEL_PATH)
    print(f"loaded model from {MODEL_PATH}, version {APP_VERSION}")


class PredictRequest(BaseModel):
    text: str

@app.get("/healthz")
def healthz():
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")
    return {"status": "ok", "version": APP_VERSION}


@app.post("/predict")
def predict(request: PredictRequest):
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")
    return {"label": _model.predict([request.text]).tolist()[0]}

Overwriting spam_app.py


In [30]:
%%writefile requirements-predictor.txt
fastapi
uvicorn[standard]
scikit-learn
joblib
pydantic

Overwriting requirements-predictor.txt


In [31]:
!docker build -t spam-api:v2 -f Dockerfile.multistage .
!minikube image load spam-api:v2


[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.1s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile.multistage            0.1s
 => => transferring dockerfile: 632B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.0s
[+] Building 0.3s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile.multistage            0.1s
 => => transferring dockerfile: 632B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.1s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.1s
[+] Building 0.4s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile.multistage            0.1s
 => => transferring dockerfile: 632B   

In [32]:
import threading
from collections import Counter

responses = []
stop = threading.Event()

def send_traffic():
    while not stop.is_set():
        try:
            r = requests.get(f"{BASE}/healthz", timeout=2)
            responses.append((r.status_code, r.json().get("version", "v1")))
        except Exception as e:
            responses.append(("ERROR", type(e).__name__))
        time.sleep(0.1)

traffic = threading.Thread(target=send_traffic)
traffic.start()
time.sleep(2)

sh("sed -i 's/image: spam-api:v1/image: spam-api:v2/' deployment.yaml")
sh("kubectl apply -f deployment.yaml")
sh('kubectl annotate deployment spam-api kubernetes.io/change-cause="v2: add version to /healthz" --overwrite')
sh("kubectl rollout status deployment/spam-api --timeout=180s")

time.sleep(3)
stop.set()
traffic.join()

print(f"requests sent during the rollout: {len(responses)}")
print("responses as (status, version):", dict(Counter(responses)))


deployment.apps/spam-api configured

deployment.apps/spam-api annotated

Waiting for deployment "spam-api" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "spam-api" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "spam-api" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "spam-api" rollout to finish: 1 old replicas are pending termination...
Waiting for deployment "spam-api" rollout to finish: 1 old replicas are pending termination...
Waiting for deployment "spam-api" rollout to finish: 1 old replicas are pending termination...
deployment "spam-api" successfully rolled out

requests sent during the rollout: 173
responses as (status, version): {(200, 'v1'): 115, (200, 'v2'): 58}


In [33]:
sh("kubectl rollout history deployment/spam-api")
sh("kubectl get replicaset -l app=spam-api")
sh("kubectl get pods -l app=spam-api -o wide")
print("healthz now:", requests.get(f"{BASE}/healthz").json());

deployment.apps/spam-api 
REVISION  CHANGE-CAUSE
1         v1: Q1 multi-stage image
2         v2: add version to /healthz


NAME                  DESIRED   CURRENT   READY   AGE
spam-api-79965dd7d4   0         0         0       7m9s
spam-api-b79996555    2         2         2       2m15s

NAME                       READY   STATUS    RESTARTS   AGE     IP            NODE           NOMINATED NODE   READINESS GATES
spam-api-b79996555-jfsm5   1/1     Running   0          2m15s   10.244.1.13   minikube-m02   <none>           <none>
spam-api-b79996555-k82dj   1/1     Running   0          2m8s    10.244.0.14   minikube       <none>           <none>

healthz now: {'status': 'ok', 'version': 'v2'}
